# Explore Programming Languages

This notebook explores the programming languages in the AIDev dataset to identify the 3 most popular languages for our library usage analysis.

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

data_dir = Path("../data")

In [ ]:
# Load the repository data
repo_pop_df = pd.read_parquet(data_dir / "repository.parquet")
pr_pop_df = pd.read_parquet(data_dir / "pull_request.parquet")

print(f"Repositories: {len(repo_pop_df):,}")
print(f"Pull Requests: {len(pr_pop_df):,}")

## Language Distribution by Repositories

In [ ]:
# Count repositories by language
lang_counts = repo_pop_df["language"].value_counts()
print("Top 10 languages by repository count:")
print(lang_counts.head(10))

# Visualize
lang_counts.head(15).plot(kind="barh", figsize=(10, 6))
plt.xlabel("Number of Repositories")
plt.ylabel("Programming Language")
plt.title("Top 15 Programming Languages in AIDev Dataset")
plt.tight_layout()
plt.show()

## Language Distribution by Pull Requests

In [ ]:
# Merge PRs with repo language
pr_with_lang = pr_pop_df.merge(
    repo_pop_df[["repository_id", "language"]], on="repository_id", how="left"
)

# Count PRs by language
pr_lang_counts = pr_with_lang["language"].value_counts()
print("\nTop 10 languages by PR count:")
print(pr_lang_counts.head(10))

# Visualize
pr_lang_counts.head(15).plot(kind="barh", figsize=(10, 6))
plt.xlabel("Number of Pull Requests")
plt.ylabel("Programming Language")
plt.title("Top 15 Programming Languages by PR Count")
plt.tight_layout()
plt.show()

## Agent Distribution by Language

In [ ]:
# Get top 3 languages
top_3_langs = pr_lang_counts.head(3).index.tolist()
print(f"\nTop 3 languages for analysis: {top_3_langs}")

# Filter to top 3 languages
top_lang_prs = pr_with_lang[pr_with_lang["language"].isin(top_3_langs)]

# Check agent distribution for each language
print("\nAgent distribution by language:")
for lang in top_3_langs:
    lang_prs = top_lang_prs[top_lang_prs["language"] == lang]
    print(f"\n{lang}: {len(lang_prs):,} PRs")
    if "agent" in lang_prs.columns:
        print(lang_prs["agent"].value_counts())

## Summary Statistics

In [ ]:
print("\n" + "=" * 60)
print("LANGUAGE ANALYSIS SUMMARY")
print("=" * 60)

for i, lang in enumerate(top_3_langs, 1):
    lang_repos = repo_pop_df[repo_pop_df["language"] == lang]
    lang_prs = top_lang_prs[top_lang_prs["language"] == lang]

    print(f"\n{i}. {lang}")
    print(f"   Repositories: {len(lang_repos):,}")
    print(f"   Pull Requests: {len(lang_prs):,}")
    print(f"   Avg PRs per repo: {len(lang_prs) / len(lang_repos):.1f}")

print(f"\nTotal PRs in top 3 languages: {len(top_lang_prs):,}")
print(f"Percentage of all PRs: {100 * len(top_lang_prs) / len(pr_pop_df):.1f}%")

In [ ]:
# Save the filtered data
top_lang_prs.to_parquet(data_dir / "top3_languages_prs.parquet", index=False)
print(f"\nSaved filtered PRs to: {data_dir / 'top3_languages_prs.parquet'}")